In [1]:
# weather + accidents

In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [23]:
spark = SparkSession.builder.appName("NYC").getOrCreate()
base_path = "/home/jovyan/work"

In [49]:
weather_df = spark.read.parquet(f"{base_path}/CLEANED_weather_partitioned/")
collision_df = spark.read.parquet(f"{base_path}/CLEANED_vehicle_collisions_partitioned/")

weather_df_filtered = weather_df.filter((weather_df.year == 2023) | (weather_df.year == 2024) | (weather_df.year == 2025))
collision_df_filtered = collision_df.filter((collision_df.year == 2023) | (collision_df.year == 2024) | (collision_df.year == 2025))

In [50]:
weather_df_filtered.sort("Date", ascending=True) #.show(3, vertical=True)

DataFrame[REPORT_TYPE: string, DailyWeather: string, HourlyDryBulbTemperature: double, HourlyPrecipitation: string, Date: date, Time: string, daily_avg_temp: double, daily_max_temp: double, daily_min_temp: double, daily_precipitation: string, daily_sunrise: double, daily_sunset: double, year: int, month: int]

In [51]:
collision_df_filtered.sort("CRASH DATE", ascending=True) #.show(3, vertical=True)

DataFrame[CRASH DATE: date, CRASH TIME: string, BOROUGH: string, LOCATION: string, ON STREET NAME: string, CROSS STREET NAME: string, OFF STREET NAME: string, NUMBER OF PERSONS INJURED: double, NUMBER OF PERSONS KILLED: double, NUMBER OF PEDESTRIANS INJURED: bigint, NUMBER OF PEDESTRIANS KILLED: bigint, NUMBER OF CYCLIST INJURED: bigint, NUMBER OF CYCLIST KILLED: bigint, NUMBER OF MOTORIST INJURED: bigint, NUMBER OF MOTORIST KILLED: bigint, CONTRIBUTING FACTOR VEHICLE 1: string, CONTRIBUTING FACTOR VEHICLE 2: string, CONTRIBUTING FACTOR VEHICLE 3: string, CONTRIBUTING FACTOR VEHICLE 4: string, CONTRIBUTING FACTOR VEHICLE 5: string, COLLISION_ID: bigint, VEHICLE TYPE CODE 1: string, VEHICLE TYPE CODE 2: string, VEHICLE TYPE CODE 3: string, VEHICLE TYPE CODE 4: string, VEHICLE TYPE CODE 5: string, year: int, month: int]

In [52]:
#get days where daily precepation >0
#TODO: compare per hour
non_rain_days_df = weather_df_filtered.filter(weather_df_filtered.daily_precipitation <= 0)
non_rain_days_df = non_rain_days_df.withColumn("weather_condition", lit("Clear"))

rain_days_df = weather_df_filtered.filter(weather_df_filtered.daily_precipitation > 0)
rain_days_df = rain_days_df.withColumn("weather_condition", lit("Rain"))

weather_tagged_df = rain_days_df.unionByName(non_rain_days_df, allowMissingColumns=True)

In [55]:
# Cuts "11:00" at ":" -> Keeps 11
collision_df_filtered = collision_df_filtered.withColumn(
    "crash_hour", 
    split(col("CRASH TIME"), ":")[0].cast("int")
)

# Cuts "23:51:00" at ":"-> Keeps 23
weather_tagged_df = weather_tagged_df.withColumn(
    "weather_hour", 
    split(col("Time"), ":")[0].cast("int")
)

joined_df = collision_df_filtered.join(
    weather_tagged_df,
    (collision_df_filtered["CRASH DATE"] == weather_tagged_df["Date"]) &
    (collision_df_filtered["crash_hour"] == weather_tagged_df["weather_hour"]),
    "inner" # Wir behalten nur Unfälle, für die wir auch das Wetter kennen
)

In [61]:
# joined_df.show(3, vertical=True)

In [57]:
# Daily avg
daily_analysis = joined_df.groupBy("weather_condition").agg(
    count("*").alias("total_collisions"),           # Zählt alle Unfälle
    countDistinct("Date").alias("amount_of_days") # Zählt die eindeutigen Tage (z.B. 65 vs 300)
).withColumn(
    "avg_collisions_per_day", 
    col("total_collisions") / col("amount_of_days") # Die magische Metrik!
)

daily_analysis.show()

+-----------------+----------------+--------------+----------------------+
|weather_condition|total_collisions|amount_of_days|avg_collisions_per_day|
+-----------------+----------------+--------------+----------------------+
|            Clear|          166676|           669|    249.14200298953662|
|             Rain|          132731|           303|     438.0561056105611|
+-----------------+----------------+--------------+----------------------+



In [60]:
# Hourly avg
hourly_analysis = joined_df.groupBy("weather_condition", "crash_hour").agg(
    count("*").alias("total_collisions"),
    countDistinct("Date").alias("amount_of_days")
).withColumn(
    "avg_collisions_per_hour", 
    col("total_collisions") / col("amount_of_days")
).orderBy("crash_hour", "weather_condition")

hourly_analysis.show(48)

+-----------------+----------+----------------+--------------+-----------------------+
|weather_condition|crash_hour|total_collisions|amount_of_days|avg_collisions_per_hour|
+-----------------+----------+----------------+--------------+-----------------------+
|            Clear|         0|            8022|           666|     12.045045045045045|
|             Rain|         0|            5938|           303|     19.597359735973598|
|            Clear|         1|            4172|           657|      6.350076103500761|
|             Rain|         1|            3300|           297|      11.11111111111111|
|            Clear|         2|            3185|           622|      5.120578778135048|
|             Rain|         2|            2632|           296|      8.891891891891891|
|            Clear|         3|            2953|           621|      4.755233494363929|
|             Rain|         3|            2548|           288|      8.847222222222221|
|            Clear|         4|            3